In [1]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.feature_selection import RFECV
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, average_precision_score
from sklearn.feature_selection import VarianceThreshold

In [2]:
# 전처리 마친 데이터 불러오기
train_df = pd.read_csv("train_e.csv")
test_df = pd.read_csv("test_e.csv")
X = train_df.drop(["임신 성공 여부"], axis=1)
y = train_df["임신 성공 여부"]

In [ ]:
# var_thresh = VarianceThreshold(threshold=0.01)
# X_var_filtered = X.loc[:, var_thresh.fit(X).get_support()]
# corr_matrix = X_var_filtered.corr().abs()
# upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
# high_corr_features = [column for column in upper.columns if any(upper[column] > 0.9)]
# X_selected = X_var_filtered.drop(columns=high_corr_features)
# X_test_selected = test_df[X_selected.columns]
# print(f"원본 특성 개수: {X.shape[1]}")
# print(f"분산 기반 선택 후 특성 개수: {X_var_filtered.shape[1]}")
# print(f"상관관계 필터링 후 최종 특성 개수: {X_selected.shape[1]}")

원본 특성 개수: 101
분산 기반 선택 후 특성 개수: 67
상관관계 필터링 후 최종 특성 개수: 52


In [ ]:
# 랜덤 포레스트 모델 생성
rf_model = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1)

# RFECV 설정 (StratifiedKFold 사용)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
selector = RFECV(estimator=rf_model, step=5, cv=cv, scoring="roc_auc", n_jobs=-1)

# RFECV 학습 및 특성 선택
selector.fit(X, y)

# 선택된 특성만 유지
X_selected = X.loc[:, selector.support_]

# 테스트 데이터에도 동일한 특성 선택 적용
X_test_selected = test_df[X_selected.columns]

# 결과 출력
print(f"원본 특성 개수: {X.shape[1]}")
print(f"선택된 특성 개수: {X_selected.shape[1]}")

In [8]:
X_train, X_val, y_train, y_val = train_test_split(X_selected, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
xgb_model = XGBClassifier(
    # colsample_bytree=0.8,
    # learning_rate=0.03,
    # max_depth=3,
    # min_child_weight=8,
    # n_estimators=600,
    # subsample=0.8,
    # scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1]),
    use_label_encoder=False, n_jobs=-1,
    eval_metric='auc', random_state=42)
xgb_model.fit(X_train, y_train)

y_val_proba = xgb_model.predict_proba(X_val)[:, 1]
y_val_pred = (y_val_proba > 0.5).astype(int)

print(classification_report(y_val, y_val_pred))
print("AUC PR:", average_precision_score(y_val, y_val_proba))
print("ROC AUC:", roc_auc_score(y_val, y_val_proba))

d:\env\envs\aimers\Lib\site-packages\xgboost\core.py:158: UserWarning: [21:43:43] WARNING: C:\b\abs_90_bwj_86a\croot\xgboost-split_1724073762025\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


              precision    recall  f1-score   support

           0       0.76      0.96      0.85     37730
           1       0.53      0.13      0.21     13211

    accuracy                           0.74     50941
   macro avg       0.64      0.55      0.53     50941
weighted avg       0.70      0.74      0.68     50941

AUC PR: 0.4460587136329267
ROC AUC: 0.7383354188274021


In [ ]:
# 전체 데이터로 재학습
model_full = xgb_model
model_full.fit(X, y)

y_pred_proba = model_full.predict_proba(test_df)[:, 1]

sample_submission = pd.read_csv('Data/sample_submission.csv')
sample_submission['probability'] = y_pred_proba
sample_submission.to_csv('./base_line.csv', index=False)

## 랜덤 서치

In [3]:
# 전처리 마친 데이터 불러오기
train_df = pd.read_csv("train_e.csv")
test_df = pd.read_csv("test_e.csv")
X = train_df.drop(["임신 성공 여부"], axis=1)
y = train_df["임신 성공 여부"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [6]:
from scipy.stats import randint, uniform

In [9]:
xgb_model = XGBClassifier(
    use_label_encoder=False, n_jobs=-1,
    eval_metric='auc', random_state=42)

param_dist = {
    'n_estimators': randint(50, 1000),
    'learning_rate': uniform(0.001, 0.299),
    'max_depth': randint(2, 11),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5),
    'gamma': uniform(0, 0.5),
    'min_child_weight': randint(1, 10)
}

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=200, 
    scoring='roc_auc', 
    cv=5,  
    verbose=2,
    n_jobs=-1,
    random_state=42
)
random_search.fit(X_train, y_train)
print("Best Parameters:", random_search.best_params_)

best_xgb = random_search.best_estimator_
y_val_proba = best_xgb.predict_proba(X_val)[:, 1]
y_val_pred = (y_val_proba > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred))
print("AUC PR:", average_precision_score(y_val, y_val_proba))
print("ROC AUC:", roc_auc_score(y_val, y_val_proba))

Fitting 5 folds for each of 200 candidates, totalling 1000 fits


d:\env\envs\aimers\Lib\site-packages\xgboost\core.py:158: UserWarning: [17:16:40] WARNING: C:\b\abs_90_bwj_86a\croot\xgboost-split_1724073762025\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Best Parameters: {'colsample_bytree': 0.8468411128907443, 'gamma': 0.1103848063943802, 'learning_rate': 0.025631932639136965, 'max_depth': 5, 'min_child_weight': 6, 'n_estimators': 508, 'subsample': 0.6366297634991047}
Accuracy: 0.7446457666712472
              precision    recall  f1-score   support

           0       0.76      0.97      0.85     37730
           1       0.54      0.11      0.19     13211

    accuracy                           0.74     50941
   macro avg       0.65      0.54      0.52     50941
weighted avg       0.70      0.74      0.68     50941

AUC PR: 0.45294441882161535
ROC AUC: 0.742064480235902


In [15]:
xgb_model = XGBClassifier(
    use_label_encoder=False, n_jobs=-1,
    eval_metric='auc', random_state=42)

param_grid = {
    'colsample_bytree': [0.8, 0.85],
    'gamma': [0.0, 0.1, 0.2],
    'learning_rate': [0.02, 0.025, 0.03],
    'max_depth': [4, 5, 6],
    'min_child_weight': [5, 6],
    'n_estimators': [500, 508],
    'subsample': [0.6, 0.65]
}

grid_search = GridSearchCV(estimator=xgb_model, 
                           param_grid=param_grid,
                           scoring='roc_auc',
                           cv=5,
                           verbose=1,
                           n_jobs=-1)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 432 candidates, totalling 2160 fits


d:\env\envs\aimers\Lib\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,
d:\env\envs\aimers\Lib\site-packages\xgboost\core.py:158: UserWarning: [20:59:03] WARNING: C:\b\abs_90_bwj_86a\croot\xgboost-split_1724073762025\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


GridSearchCV(cv=5,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False,
                                     eval_metric='auc', feature_types=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=None,
                                     learning_rate=None...
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=-1, num_parallel_tree=None,
                                     random_state=42, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.8, 0.85],
                         'gamma': [0.0, 0.1, 0.2],
                         'learning_rate': [0.02, 0.025, 0.03],
                         'max_depth': [4, 5, 6], 'min_child_weight': [5, 6],
                         'n_estimators': [500, 508], 'subsample': [0.6, 0.65]},
             scoring='roc_auc', verbose=1)

In [18]:
print("Best Parameters:", grid_search.best_params_)
best_xgb = grid_search.best_estimator_
y_train_proba = best_xgb.predict_proba(X_train)[:, 1]
y_train_pred = (y_train_proba > 0.5).astype(int)

print(classification_report(y_train, y_train_pred))
print("AUC PR:", average_precision_score(y_train, y_train_proba))
print("ROC AUC:", roc_auc_score(y_train, y_train_proba))

y_val_proba = grid_search.predict_proba(X_val)[:, 1]
y_val_pred = (y_val_proba > 0.5).astype(int)

print(classification_report(y_val, y_val_pred))
print("AUC PR:", average_precision_score(y_val, y_val_proba))
print("ROC AUC:", roc_auc_score(y_val, y_val_proba))

Best Parameters: {'colsample_bytree': 0.85, 'gamma': 0.0, 'learning_rate': 0.025, 'max_depth': 5, 'min_child_weight': 6, 'n_estimators': 500, 'subsample': 0.65}
              precision    recall  f1-score   support

           0       0.76      0.97      0.85    150921
           1       0.58      0.12      0.20     52841

    accuracy                           0.75    203762
   macro avg       0.67      0.54      0.52    203762
weighted avg       0.71      0.75      0.68    203762

AUC PR: 0.470119287447499
ROC AUC: 0.7470945057390643
              precision    recall  f1-score   support

           0       0.76      0.97      0.85     37730
           1       0.54      0.11      0.19     13211

    accuracy                           0.75     50941
   macro avg       0.65      0.54      0.52     50941
weighted avg       0.70      0.75      0.68     50941

AUC PR: 0.4536198859536606
ROC AUC: 0.7421310173639324


In [ ]:
model_full = grid_search.best_estimator_
model_full.fit(X, y)

y_pred_proba = model_full.predict_proba(test_df)[:, 1]

sample_submission = pd.read_csv('Data/sample_submission.csv')
sample_submission['probability'] = y_pred_proba
sample_submission.to_csv('./rgb.csv', index=False)

d:\env\envs\aimers\Lib\site-packages\xgboost\core.py:158: UserWarning: [15:36:21] WARNING: C:\b\abs_90_bwj_86a\croot\xgboost-split_1724073762025\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [ ]:
from sklearn.linear_model import LogisticRegression